# CEREBUS Markov Chain Market State Model

Learns state transition probabilities from Holy Grail data + price sequences.

States (from Holy Grail ontology):
- RESET → AR_SET → P90_FIRED → T1/T2/T3_ACTIVE → TARGET_25/50/100 → STALL_ZONE → DEEP_STATE → REKEY → FAILURE → HARD_EXIT

Uses Holy Grail priors (98.22% for -25%, 96.44% for -50%) as starting points.
Learns refined probabilities from actual price data across all 18 assets.

In [ ]:
# Install dependencies
!pip install -q pandas numpy pyarrow matplotlib seaborn
print('Done')

In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/larger-lab/quant-lab/ml/data/training')
OUTPUT_DIR = Path('/content/drive/MyDrive/larger-lab/quant-lab/ml/data/markov_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Drive mounted')

In [ ]:
# Load all training data
files = sorted(DATA_DIR.glob('*_training.parquet'))
print(f'Found {len(files)} asset files')

all_data = {}
for f in files:
    symbol = f.stem.replace('_training', '')
    df = pd.read_parquet(f)
    all_data[symbol] = df
    print(f'  {symbol}: {len(df):,} rows x {len(df.columns)} cols')

In [ ]:
# Define market states (from Holy Grail ontology)
STATES = [
    'RESET', 'AR_SET', 'P90_FIRED', 'T1_ACTIVE', 'T2_ACTIVE', 'T3_ACTIVE',
    'TARGET_25', 'TARGET_50', 'TARGET_100', 'STALL_ZONE', 'DEEP_STATE',
    'REKEY', 'REKEY_CONSOLID', 'REKEY_EXTENSION', 'FAILURE', 'HARD_EXIT', 'REGIME_FLIP'
]
STATE_IDX = {s: i for i, s in enumerate(STATES)}
N_STATES = len(STATES)
print(f'{N_STATES} states defined')

In [ ]:
# Holy Grail prior probabilities (from extracted data)
# These are the known transition probabilities from the Holy Grail
HOLY_GRAIL_PRIORS = {
    ('RESET', 'AR_SET'): 1.0,        # Every session starts with AR
    ('AR_SET', 'P90_FIRED'): 0.95,   # 95% of sessions get a P90
    ('P90_FIRED', 'T1_ACTIVE'): 0.40,  # ~40% are T1 (<20p AR)
    ('P90_FIRED', 'T2_ACTIVE'): 0.35,  # ~35% are T2 (20-30p AR)
    ('P90_FIRED', 'T3_ACTIVE'): 0.25,  # ~25% are T3 (30-45p AR)
    ('T1_ACTIVE', 'TARGET_25'): 0.982,  # -25% hit rate (T1)
    ('T2_ACTIVE', 'TARGET_25'): 0.964,  # -25% hit rate (T2)
    ('T3_ACTIVE', 'TARGET_25'): 0.922,  # -25% hit rate (T3)
    ('TARGET_25', 'TARGET_50'): 0.964,  # -50% hit rate after -25%
    ('TARGET_50', 'TARGET_100'): 0.922, # -100% hit rate after -50%
    ('TARGET_100', 'REKEY'): 0.715,     # 132% violation rate
    ('REKEY', 'REKEY_CONSOLID'): 0.850,  # 85% consolidation (12-24h)
    ('REKEY_CONSOLID', 'REKEY_EXTENSION'): 0.780,  # -50% extension target
    ('TARGET_25', 'STALL_ZONE'): 0.342,  # 34.2% reach stall zone
    ('STALL_ZONE', 'REVERSAL'): 0.042,   # 4.2% full reversal after stall
    ('STALL_ZONE', 'CONTINUATION'): 0.860,  # 86% continue after stall
    ('FAILURE', 'SOFT'): 0.642,          # Type 1: Soft failure (midpoint only)
    ('FAILURE', 'INTERNAL_RESET'): 0.249, # Type 2: Same-side recycle
    ('FAILURE', 'REGIME_FLIP'): 0.109,   # Type 3: Opposite-side confirmed
}
print(f'{len(HOLY_GRAIL_PRIORS)} prior transitions loaded')

In [ ]:
# Learn transition probabilities from actual data
def learn_transitions_from_data(all_data, states):
    """Learn state transition probabilities from training data labels."""
    transition_counts = defaultdict(lambda: defaultdict(int))
    state_counts = defaultdict(int)
    
    for symbol, df in all_data.items():
        # Filter to session-start bars (Monday activation window)
        if 'hour_est' in df.columns:
            mask = (df['hour_est'] >= 3) & (df['hour_est'] < 5)
            if 'day_of_week' in df.columns:
                mask = mask & (df['day_of_week'] == 0)
            sessions = df[mask]
        else:
            sessions = df
        
        for _, row in sessions.iterrows():
            # Determine current state from labels